# Inventory & Supply Chain Management Dashboard

**Import Required Python Libraries**

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import mysql.connector

In [ ]:
# Connect Python to MySQL Database
conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="#Junaid@xxxx",
    database="userinterface"
)

In [7]:
# Check Database Connection Status
conn.is_connected()

True

In [8]:
# Create Database Cursor Object
cursor=conn.cursor()

In [ ]:
# Create Reusable Database Connection Function
def connect_to_df():
    return mysql.connector.connect(
    host="localhost",
    user="root",
    password="JunaidXXXX"
    database="userinterface")
    
    

In [10]:
# Test Database Connection Function
connect_to_df().is_connected()

True

In [11]:
# Creates cursor that returns data in dictionary format.
db=connect_to_df()
cursor=db.cursor(dictionary= True)


# Define KPI SQL Queries

In [12]:
#Stores SQL queries for business metrics and dashboard KPIs.
queries={
    "Total Suppliers": "select count(*) as Total_suppliers from suppliers",
   
    "Total Product": "select count(*) as Total_Product from products",
    
    "Total Categories": "Select count(distinct category) as Total_Category from products",
    
    "Total Sale Value(Last three month)": """select round(sum(abs(s.change_quantity)* p.price),2) as Total_Sales from stock_entries s
    join products p on p.product_id=s.product_id where change_type= "Sale"
    and entry_date>= (select date_sub(max(se.entry_date), Interval 3 month) from stock_entries se)""",
    
    "Total Restock Value(Last three month)": """select round(sum(abs(s.change_quantity)* p.price),2) as Total_Sales from stock_entries s
    join products p on p.product_id=s.product_id where change_type= "Restock"
    and entry_date>=(select date_sub(max(se.entry_date), Interval 3 month) from stock_entries se)""",

    "Below Reorder & No pending order":"""select count(*) from products where stock_quantity<reorder_level 
    and product_id not in
    (
    select product_id from reorders where status="Pending"
    )"""
}

    
    
    


In [13]:
#Execute KPI Queries and Store Results
result={}
for label, query in queries.items():
    cursor.execute(query)
    row=cursor.fetchone()
    result[label]=list(row.values())[0]
    

In [14]:
result

{'Total Suppliers': 50,
 'Total Product': 202,
 'Total Categories': 5,
 'Total Sale Value(Last three month)': None,
 'Total Restock Value(Last three month)': 452611.11,
 'Below Reorder & No pending order': 15}

In [15]:
# Creates reusable function for dashboard KPI calculations.
def basic_information(cursor):
    queries={
    "Total Suppliers": "select count(*) as Total_suppliers from suppliers",
   
    "Total Product": "select count(*) as Total_Product from products",
    
    "Total Categories": "Select count(distinct category) as Total_Category from products",
    
    "Total Sale Value(Last one year)": """select round(sum(abs(s.change_quantity)* p.price),2) as Total_Sales from stock_entries s
    join products p on p.product_id=s.product_id where change_type= "Sale"
    and entry_date>= (select date_sub(max(se.entry_date), Interval 1 year) from stock_entries se)""",
    
    "Total Restock Value(Last three month)": """select round(sum(abs(s.change_quantity)* p.price),2) as Total_Sales from stock_entries s
    join products p on p.product_id=s.product_id where change_type= "Restock"
    and entry_date>=(select date_sub(max(se.entry_date), Interval 3 month) from stock_entries se)""",

    "Below Reorder & No pending order":"""select count(*) from products where stock_quantity<reorder_level 
    and product_id not in
    (
    select product_id from reorders where status="Pending"
    )"""}

    result={}
    for label, query in queries.items():
        cursor.execute(query)
        row=cursor.fetchone()
        result[label]=list(row.values())[0]


    return result

    

In [16]:
# Extracts KPI names/keys from dictionary
get_basic=basic_information(cursor)
keys=list(get_basic.keys())
keys

['Total Suppliers',
 'Total Product',
 'Total Categories',
 'Total Sale Value(Last one year)',
 'Total Restock Value(Last three month)',
 'Below Reorder & No pending order']

In [17]:
# Extracts KPI values from dictionary.
get_basic=basic_information(cursor)
keys=list(get_basic.values())
keys

[50, 202, 5, 410600.09, 452611.11, 15]

In [19]:
# Stores queries for supplier and inventory reports.
queries={
    "Supplier Details": "select supplier_name, contact_name, email, phone from suppliers",
    "Product Detail with suppliers": """select p.product_name, p.stock_quantity, p.reorder_level, s.supplier_name from products p 
    join suppliers s on p.supplier_id=s.supplier_id order by p.product_name""",
    "Product need reorder": """select product_id, product_name, stock_quantity, reorder_level from products where stock_quantity<=reorder_level"""
}

tables={}
for label, query in queries.items():
    cursor.execute(query)
    tables[label]=cursor.fetchall()
    

In [24]:
# Create Additional Tables Function
def get_additional_tables(cursor):
    queries={
    "Supplier Details": "select supplier_name, contact_name, email, phone from suppliers",
    "Product Detail with suppliers": """select p.product_name, p.stock_quantity, p.reorder_level, s.supplier_name from products p 
    join suppliers s on p.supplier_id=s.supplier_id order by p.product_name""",
    "Product need reorder": """select product_id, product_name, stock_quantity, reorder_level from products where stock_quantity<=reorder_level"""}

    tables={}
    for label, query in queries.items():
        cursor.execute(query)
        tables[label]=cursor.fetchall()
        
    return tables

In [15]:
#Call new procedure
def Add_new_product(cursor, db, p_name, p_category, p_price, p_stock, p_reorder,  p_suppliers):
    proc_call="call AddNewProduct_id(%s, %s, %s, %s, %s, %s)"
    params= (p_name, p_category, p_price, p_reorder,  p_stock,  p_suppliers)
    cursor.execute(proc_call, params)
    db.commit()
    

In [16]:
# Retrieves distinct product categories from database.
def get_categories(cursor):
    cursor.execute("select distinct category from products order by category asc")
    rows=cursor.fetchall()
    return [row["category"] for row in rows]
    
    

In [20]:
# Retrieves supplier IDs and names from database.
def get_suppliers(cursor):
    cursor.execute("select supplier_id, supplier_name from suppliers order by supplier_name asc")
    return cursor.fetchall()
    

In [22]:
# Retrieves all products and inventory tracking data.
def get_all_product(cursor):
    cursor.execute("select product_id, product_name from products order by product_name")
    rows=cursor.fetchall()
    return rows

In [29]:
# Fetches historical inventory records for selected product.
def get_product_history(cursor, product_id):
    query=("select * from Inventory_history where product_id = %s order by record_date desc")
    cursor.execute(query, (product_id,))
    return cursor.fetchall()
    


In [13]:
# Create Reorder Placement Function
def place_reorder(cursor, db, product_id, reorder_quantity):
    query= """
    Insert into reorders (reorder_id, product_id, reorder_quantity, reorder_date, status)
    select ifnull(max(reorder_id),0)+1, 101, 100, curdate(), "ordered" from reorders
    """
    cursor.excute(query, (product_id, reorder_quantity,))
    db.commit()
    
    

In [26]:
# Retrieves all pending reorder requests
def get_pending_reorder(cursor):
    cursor.execute("select r.reorder_id, p.product_name from reorders r join products p on r.product_id=p.product_id")
    return cursor.fetchall()

In [27]:
# Marks reorder as received using stored procedure.
def received_orders(cursor, db, reorder_id):
    cursor.callproc("Order_received",[reorder_id])
    db.commit()
    
    

*_Finish_*